# DeepSeek 在线模型调用

这个 Notebook 对比底层 OpenAI 兼容 SDK、推荐的 `ChatDeepSeek` 和兼容写法 `ChatOpenAI`。执行包含 `invoke` 的单元会访问真实 DeepSeek API，并可能产生费用。

In [ ]:
import os

from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment()
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_API_BASE", "https://api.deepseek.com")

## 1. 使用 OpenAI SDK 直接调用 DeepSeek

这种方式最接近 HTTP API，适合理解 LangChain 模型适配器下层发生了什么。

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url)
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful translator."},
        {"role": "user", "content": "把“你好”翻译成日语。"},
    ],
)
print(response.choices[0].message.content)

## 2. 使用 ChatDeepSeek（推荐）

`ChatDeepSeek` 能保留 DeepSeek 特有响应，并提供 LangChain 的 invoke、stream、batch、async、工具调用和结构化输出接口。

In [ ]:
from langchain_deepseek import ChatDeepSeek

deepseek_model = ChatDeepSeek(
    model="deepseek-flash",
    temperature=0,
    timeout=30,
    max_retries=2,
    api_key=api_key,
    base_url=base_url,
)
messages = [
    ("system", "You are a helpful translator. Translate the user sentence to Japanese."),
    ("human", "你好"),
]
print(deepseek_model.invoke(messages).content)

## 3. 使用 ChatOpenAI 兼容接口

DeepSeek 兼容 OpenAI Chat Completions API，因此可以这样调用；正式 LangChain 项目仍优先使用 `ChatDeepSeek`，避免丢失提供商特有字段。

In [ ]:
from langchain_openai import ChatOpenAI

compatible_model = ChatOpenAI(
    model="deepseek-flash",
    api_key=api_key,
    base_url=base_url,
    temperature=0,
    timeout=30,
    max_retries=2,
)
print(compatible_model.invoke("Hello").content)